In [2]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory


In [3]:
df = pd.read_excel('exerc6.xlsx', index_col='maquinas')

In [4]:
df

,custo_fixo,custo_variavel,capacidade
maquinas,,,
1,1000,21,500
2,950,23,600
3,875,25,750
4,850,24,400
5,800,20,600
6,700,26,800


In [5]:
model = pyo.ConcreteModel()

model.maquinas = pyo.Set(initialize=df.index.tolist())
model.custo_fixo = pyo.Param(model.maquinas, initialize=df['custo_fixo'].to_dict())
model.custo_variavel = pyo.Param(model.maquinas, initialize=df['custo_variavel'].to_dict())
model.capacidade = pyo.Param(model.maquinas, initialize=df['capacidade'].to_dict())
model.x = pyo.Var(model.maquinas, domain=pyo.NonNegativeIntegers)

In [6]:
#objetivo

def obj(model):
    return sum(model.custo_fixo[m] + model.custo_variavel[m] * model.x[m] for m in model.maquinas)

model.obj = pyo.Objective(rule=obj, sense=pyo.minimize)

In [7]:
#restrição de demanda
def demanda1(model):
    return sum(model.x[m] for m in model.maquinas) == 1800
model.demanda = pyo.Constraint(rule=demanda1)

#restrição de capacidade
def capacidade1(model, m):
    return model.x[m] <= model.capacidade[m]
model.capacidade_model = pyo.Constraint(model.maquinas, rule=capacidade1)

In [8]:
model.pprint()

1 Set Declarations
    maquinas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    6 : {1, 2, 3, 4, 5, 6}

3 Param Declarations
    capacidade : Size=6, Index=maquinas, Domain=Any, Default=None, Mutable=False
        Key : Value
          1 :   500
          2 :   600
          3 :   750
          4 :   400
          5 :   600
          6 :   800
    custo_fixo : Size=6, Index=maquinas, Domain=Any, Default=None, Mutable=False
        Key : Value
          1 :  1000
          2 :   950
          3 :   875
          4 :   850
          5 :   800
          6 :   700
    custo_variavel : Size=6, Index=maquinas, Domain=Any, Default=None, Mutable=False
        Key : Value
          1 :    21
          2 :    23
          3 :    25
          4 :    24
          5 :    20
          6 :    26

1 Var Declarations
    x : Size=6, Index=maquinas
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          1 :     0 : 

In [9]:
res = SolverFactory('cplex')
opt = res.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmp7hwvrsmn.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmpbjk6x4vf.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmpbjk6x4vf.pyomo.lp
Objective sense      : Minimize
Variables            :       7  [Fix: 1,  General Integer: 6]
Objective nonzeros   :       7
Linear constraints   :       7  [Less: 6,  Equal: 1]
  Nonzeros           :      12
  RHS nonzeros       :       7

Variables            : Min LB: 0.000000         Max UB: 1.000000       
Objective nonzeros   : Min

In [15]:
for re in model.x:
    print(f'Maquina {re}: {model.x[re].value}')

print(f'Custo total: R$ {model.obj():.2f}')

Maquina 1: 500.0
Maquina 2: 600.0
Maquina 3: 0.0
Maquina 4: 100.0
Maquina 5: 600.0
Maquina 6: 0.0
Custo total: R$ 43875.00
